<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 4 &middot; Part 3 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Forward Propagation in NumPy

Part 2 let TensorFlow find the weights and run the network. This part takes the library away.
You will write forward propagation yourself &mdash; first one unit at a time, then as a loop,
then as a single matrix product &mdash; and check every version against the others and against
TensorFlow.

There is a second reason to do this. Writing the network by hand lets you *design* one by hand.
Section 4 builds a coffee-roasting classifier with no training at all, choosing every weight
from the geometry of the problem, and it works. After that, a trained network's weights stop
looking like magic.

**Companion theory lecture:** CSE 815, Week 4, Part 2 (Neural Networks in Code), sections
"Forward Propagation from Scratch" and "Vectorisation and Matrix Multiplication".

## What you will be able to do

1. Compute one layer unit by unit, with variable names that follow the lecture's notation.
2. Write a general `my_dense` and `my_sequential` for any fully connected sigmoid network.
3. Design weights by hand so that each hidden unit detects one side of a region, and the output
   unit combines them.
4. Confirm that your implementation and TensorFlow compute the same function.
5. Multiply matrices correctly: shapes, the transpose, and `@`.
6. Replace the loop over units with one matrix product, verify they agree, and measure the
   speed-up.

---

## 1. Setup

The same roasting data as Part 2.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

def load_coffee(seed=816, m=200):
    """Roasting trials: temperature (deg C) and duration (min); y = 1 for a good roast.
    A roast is good inside the triangle with corners (175, 12), (175, 15) and (260, 12):
    hot enough, long enough, and not so hot for so long that it burns."""
    rng = np.random.default_rng(seed)
    temp = np.round(rng.uniform(150, 285, m), 1)
    dur = np.round(rng.uniform(11.5, 15.5, m), 2)
    good = (temp > 175) & (dur > 12) & (3 * temp + 85 * dur < 1800)
    return np.column_stack([temp, dur]), good.astype(float)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


X_cof, y_cof = load_coffee()
print("X_cof.shape:", X_cof.shape, f"  good roasts: {int(y_cof.sum())} of {len(y_cof)}")

---

## 2. One layer, unit by unit

Start with the lecture's worked example, so that every number can be checked against the board.
Variable names follow the notation: `w1_2` is $\mathbf{w}^{[1]}_2$, the weights of layer 1,
unit 2.

In [ ]:
x = np.array([1.0, 2.0])

w1_1 = np.array([1.0, 2.0]);   b1_1 = -1.0
z1_1 = np.dot(w1_1, x) + b1_1
a1_1 = sigmoid(z1_1)

w1_2 = np.array([-3.0, 4.0]);  b1_2 = 1.0
z1_2 = np.dot(w1_2, x) + b1_2
a1_2 = sigmoid(z1_2)

w1_3 = np.array([5.0, -6.0]);  b1_3 = 2.0
z1_3 = np.dot(w1_3, x) + b1_3
a1_3 = sigmoid(z1_3)

a1 = np.array([a1_1, a1_2, a1_3])
print("z1 =", np.array([z1_1, z1_2, z1_3]))
print("a1 =", a1)

w2_1 = np.array([2.0, -1.0, -3.0]);  b2_1 = 0.5
a2_1 = sigmoid(np.dot(w2_1, a1) + b2_1)
print("a2 =", round(float(a2_1), 4))

assert np.allclose(a1, [0.9820, 0.9975, 0.0067], atol=5e-5)
assert np.isclose(a2_1, 0.8094, atol=5e-5)

Twelve lines, and the only thing that changes from one unit to the next is the parameters. That
is a loop waiting to be written.

---

## 3. A general dense layer, and a network

Stack each unit's weight vector as a **column** of $\mathbf{W}$ &mdash; the same convention
TensorFlow uses &mdash; so that unit $j$'s weights are `W[:, j]`:

In [ ]:
def my_dense(a_in, W, b):
    """One fully connected sigmoid layer, one unit at a time.
    a_in: (n_in,) activations of the previous layer
    W:    (n_in, units), column j holds unit j's weights
    b:    (units,)
    returns (units,)"""
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        z = np.dot(W[:, j], a_in) + b[j]
        a_out[j] = sigmoid(z)
    return a_out


def my_sequential(x, params):
    """Forward propagation through any number of layers. params = [(W1, b1), (W2, b2), ...]"""
    a = x
    for W, b in params:
        a = my_dense(a, W, b)
    return a


W1 = np.array([[1., -3., 5.],
               [2., 4., -6.]])
b1 = np.array([-1., 1., 2.])
W2 = np.array([[2.], [-1.], [-3.]])
b2 = np.array([0.5])
lecture_net = [(W1, b1), (W2, b2)]

for xi in [np.array([1., 2.]), np.array([-1., -1.])]:
    print(xi, "->", my_sequential(xi, lecture_net))

assert np.allclose(my_sequential(np.array([1., 2.]), lecture_net), [0.8094], atol=5e-5)
assert np.allclose(my_sequential(np.array([-1., -1.]), lecture_net), [0.0562], atol=5e-5)
assert np.allclose(my_dense(x, W1, b1), a1)
print("\nThe loop reproduces section 2, and the lecture's second example too.")

`my_sequential` is a complete inference engine for sigmoid networks of any depth: it is what
`model.predict` does, minus speed and minus every activation other than the sigmoid.

---

## 4. Designing a network by hand

The good-roast region is a triangle, and a triangle is the intersection of three half-planes:

| unit | the roast is good only if | as a linear inequality |
|---|---|---|
| 1 | it is hot enough | $t - 175 > 0$ |
| 2 | it is long enough | $d - 12 > 0$ |
| 3 | it is not too hot for too long | $1800 - 3t - 85d > 0$ |

(The third line passes through $(175, 15)$ and $(260, 12)$; check it.)

A sigmoid unit with $z = k \cdot (\text{left-hand side})$ outputs nearly 1 on the good side of
its line and nearly 0 on the other, more sharply the larger $k$ is. So one hidden unit per
inequality turns each into a soft yes/no. The output unit must then say yes only when **all
three** say yes: a soft AND, $g\bigl(10(a_1 + a_2 + a_3) - 25\bigr)$, which exceeds $\tfrac12$
only when the sum of the activations exceeds $2.5$.

In [ ]:
# Scale each inequality so that z changes by a few units across a degree or a minute.
# Unit 1: 0.5 (t - 175)          Unit 2: 5 (d - 12)          Unit 3: 0.06 (1800 - 3t - 85d)
W1_hand = np.array([[0.5, 0.0, -0.18],
                    [0.0, 5.0, -5.1]])
b1_hand = np.array([-87.5, -60.0, 108.0])

# Output: a soft AND of the three detectors.
W2_hand = np.array([[10.], [10.], [10.]])
b2_hand = np.array([-25.])
hand_net = [(W1_hand, b1_hand), (W2_hand, b2_hand)]

# Check the third unit's line really passes through both corners: z must be 0 there.
for corner in [(175, 15), (260, 12)]:
    z3 = np.dot(W1_hand[:, 2], corner) + b1_hand[2]
    print(f"z3 at {corner} = {z3:+.4f}")
    assert np.isclose(z3, 0.0)

print(f"\n{'temp':>6} {'dur':>6} {'a1':>7} {'a2':>7} {'a3':>7} {'P(good)':>8}")
print("-" * 46)
for xi in [(200, 13.9), (200, 17.0), (230, 12.5), (170, 13.0), (250, 14.0)]:
    a_hid = my_dense(np.array(xi, dtype=float), W1_hand, b1_hand)
    p = my_dense(a_hid, W2_hand, b2_hand)[0]
    print(f"{xi[0]:6.0f} {xi[1]:6.1f} {a_hid[0]:7.4f} {a_hid[1]:7.4f} {a_hid[2]:7.4f} {p:8.4f}")

Read the rows. The roast at $(200, 17.0)$ is long enough and hot enough &mdash; units 1 and 2
both say yes &mdash; but unit 3 says it is burnt, and the AND follows unit 3. The roast at
$(170, 13.0)$ fails unit 1. Each hidden unit is a feature with a meaning you chose: exactly the
"affordability, awareness, perceived quality" story from the lecture, except that here the
story is true by construction.

In [ ]:
p_hand = np.array([my_sequential(xi, hand_net)[0] for xi in X_cof])
pred_hand = (p_hand >= 0.5).astype(float)
acc_hand = (pred_hand == y_cof).mean()
print(f"hand-designed network: accuracy {acc_hand:.4f} on all {len(y_cof)} roasts")

wrong = np.flatnonzero(pred_hand != y_cof)
for i in wrong:
    t, d = X_cof[i]
    a_hid = my_dense(X_cof[i], W1_hand, b1_hand)
    print(f"  misclassified: t = {t}, d = {d}, y = {y_cof[i]:.0f}, P(good) = {p_hand[i]:.4f}")
    print(f"  hidden activations {a_hid}, sum {a_hid.sum():.4f} -- the AND needs 2.5")

assert acc_hand >= 0.99

The one mistake is a good roast in the corner of the triangle: one degree above the
temperature edge and a third of a minute above the duration edge. A sigmoid with finite $k$ is
not a hard threshold, so units 1 and 2 are only *fairly* sure (about 0.62 and 0.82), and the two
doubts together pull the sum below the AND's threshold of 2.5. Sharpen unit 1 &mdash; multiply
its weights and bias by 10 &mdash; and the point is classified correctly (exercise 1 asks you to
try each unit). No training, no gradient descent: just the geometry, turned into weights.

In [ ]:
def plot_units(params, title):
    """Each hidden unit's activation over the input plane, plus the network output."""
    t = np.linspace(150, 285, 200)
    d = np.linspace(11.5, 15.5, 200)
    T, D = np.meshgrid(t, d)
    grid = np.column_stack([T.ravel(), D.ravel()])
    (W1_, b1_), (W2_, b2_) = params
    A1 = sigmoid(grid @ W1_ + b1_)
    A2 = sigmoid(A1 @ W2_ + b2_)
    panels = [A1[:, j] for j in range(A1.shape[1])] + [A2[:, 0]]
    names = [f"hidden unit {j + 1}" for j in range(A1.shape[1])] + ["output"]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.0))
    for ax, P, name in zip(axes, panels, names):
        ax.contourf(T, D, P.reshape(T.shape), levels=np.linspace(0, 1, 11), cmap="Blues")
        ax.contour(T, D, P.reshape(T.shape), levels=[0.5], colors=[AMBER], linewidths=1.5)
        good = y_cof == 1
        ax.scatter(X_cof[good, 0], X_cof[good, 1], s=8, c=TEAL, marker="s")
        ax.scatter(X_cof[~good, 0], X_cof[~good, 1], s=8, c=ROSE, marker="x")
        ax.set_title(name, fontsize=9)
        ax.set_xticks([175, 260])
        ax.set_yticks([12, 15])
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


plot_units(hand_net, "Hand-designed: each hidden unit is one edge of the triangle")

---

## 5. Does TensorFlow agree?

Load the hand-designed weights into a TensorFlow model with the same architecture and compare
outputs on all 200 roasts.

In [ ]:
tf_net = Sequential([
    Input(shape=(2,)),
    Dense(units=3, activation="sigmoid", name="layer1"),
    Dense(units=1, activation="sigmoid", name="layer2"),
])
tf_net.get_layer("layer1").set_weights([W1_hand, b1_hand])
tf_net.get_layer("layer2").set_weights([W2_hand, b2_hand])

p_tf = tf_net.predict(X_cof, verbose=0)[:, 0]
diff = np.abs(p_tf - p_hand).max()
print(f"largest |TensorFlow - mine| over 200 roasts: {diff:.2e}")

assert np.allclose(p_tf, p_hand, atol=1e-5)
assert np.array_equal(p_tf >= 0.5, p_hand >= 0.5)
print("Same function. TensorFlow's Dense layer is my_dense, in float32.")

---

## 6. Matrix multiplication

The loop in `my_dense` computes one dot product per unit. A matrix product computes all of them
&mdash; for all examples &mdash; at once. Before using it, make sure the rules are automatic.

If $\mathbf{A}$ is $m \times n$ and $\mathbf{B}$ is $n \times p$, then $\mathbf{AB}$ is
$m \times p$, and entry $(i, j)$ is **row $i$ of $\mathbf{A}$ dotted with column $j$ of
$\mathbf{B}$**:

$$(\mathbf{AB})_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}$$

The inner dimensions must match. This is the lecture's example.

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 0, 1],
              [-1, 2, 0]])

AB = A @ B                          # the same as np.matmul(A, B)
print("A @ B =\n", AB, "\nshape", AB.shape)
assert np.array_equal(AB, [[3, 4, 1], [11, 8, 3]])

# Entry by entry, from the definition.
manual = np.zeros((2, 3), dtype=int)
for i in range(2):
    for j in range(3):
        manual[i, j] = np.dot(A[i, :], B[:, j])
assert np.array_equal(manual, AB)
print("\n(AB)[1, 0] = row 1 of A . column 0 of B =", A[1], ".", B[:, 0], "=", AB[1, 0])

# The transpose swaps rows and columns.
print("\nA.T =\n", A.T)
assert np.array_equal(A.T, [[1, 3], [2, 4]])

In [ ]:
# Order matters, and the shapes say so before any arithmetic happens.
try:
    B @ A                           # (2, 3) @ (2, 2): inner dimensions 3 and 2
except ValueError as exc:
    print("B @ A -> ValueError:", str(exc).splitlines()[0][:90])

# Even when both orders are defined, they are not equal.
C = np.array([[0, 1], [1, 0]])
print("\nA @ C =\n", A @ C, "\nC @ A =\n", C @ A)
assert not np.array_equal(A @ C, C @ A)

# And '*' is not matrix multiplication: it multiplies element by element.
print("\nA * C =\n", A * C, "   <- elementwise, a different operation entirely")

> **`*` versus `@`.** `A * B` multiplies matching entries and needs equal (or broadcastable)
> shapes. `A @ B` is the matrix product and needs matching *inner* dimensions. Code that uses
> `*` where `@` was meant sometimes runs without error and is simply wrong.

---

## 7. Vectorised forward propagation

Put the examples in the rows of $\mathbf{A}_{\text{in}}$, shape `(m, n_in)`. Then

$$\mathbf{Z} = \mathbf{A}_{\text{in}}\mathbf{W} + \mathbf{b}, \qquad
  \mathbf{A}_{\text{out}} = g(\mathbf{Z})$$

with shapes $(m, n_{\text{in}})(n_{\text{in}}, \text{units}) \to (m, \text{units})$. Row $i$ of
$\mathbf{Z}$ belongs to example $i$, column $j$ to unit $j$. The bias, shape `(units,)`, is
broadcast to every row.

In [ ]:
def my_dense_v(A_in, W, b):
    """Vectorised dense layer. A_in: (m, n_in), W: (n_in, units), b: (units,) -> (m, units)"""
    return sigmoid(A_in @ W + b)


def my_sequential_v(X, params):
    A = X
    for W, b in params:
        A = my_dense_v(A, W, b)
    return A


# The lecture's matrix slide: two examples through the 2 -> 3 -> 1 network.
X_demo = np.array([[1., 2.],
                   [-1., -1.]])
Z1 = X_demo @ W1 + b1
print("Z1 =\n", Z1)
assert np.array_equal(Z1, [[4, 6, -5], [-4, 0, 3]])
A2 = my_sequential_v(X_demo, lecture_net)
print("A2 =", A2.ravel())
assert np.allclose(A2.ravel(), [0.8094, 0.0562], atol=5e-5)

# Loop and matrix product must agree on every roast, for the hand-designed network.
P_loop = np.array([my_sequential(xi, hand_net) for xi in X_cof])
P_vec = my_sequential_v(X_cof, hand_net)
assert P_vec.shape == (200, 1)
assert np.allclose(P_loop, P_vec)
print("\nLoop and vectorised versions agree on all 200 roasts.")

In [ ]:
import time

rng = np.random.default_rng(0)
X_big = rng.uniform([150, 11.5], [285, 15.5], size=(20000, 2))
wide_net = [(rng.normal(size=(2, 64)), rng.normal(size=64)),
            (rng.normal(size=(64, 32)) / 8, rng.normal(size=32)),
            (rng.normal(size=(32, 1)) / 6, rng.normal(size=1))]
X_big_s = (X_big - X_big.mean(axis=0)) / X_big.std(axis=0)

t0 = time.perf_counter()
P_loop = np.array([my_sequential(xi, wide_net) for xi in X_big_s])
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
P_vec = my_sequential_v(X_big_s, wide_net)
t_vec = time.perf_counter() - t0

assert np.allclose(P_loop, P_vec)
print(f"20,000 examples through a 2 -> 64 -> 32 -> 1 network")
print(f"  loop over examples and units: {t_loop:8.3f} s")
print(f"  matrix products             : {t_vec:8.4f} s")
print(f"  speed-up                    : about {t_loop / t_vec:,.0f}x")
print("\nThe exact figures depend on your machine; the order of magnitude does not.")

The arithmetic is identical. The difference is that `@` hands the whole computation to compiled
linear-algebra code that uses every core and the processor's vector instructions, while the
Python loop pays interpreter overhead on every one of the roughly two million dot products. On a
GPU the gap is larger still. This &mdash; not any new mathematics &mdash; is why networks with
millions of parameters are practical.

### Checkpoint 1

The XOR function is 1 when exactly one of two binary inputs is 1. No single sigmoid unit can
compute it (no line separates $\{(0,1),(1,0)\}$ from $\{(0,0),(1,1)\}$), but a $2 \to 2 \to 1$
network can, because

$$x_1 \text{ XOR } x_2 = (x_1 \text{ OR } x_2) \text{ AND } \text{NOT}(x_1 \text{ AND } x_2).$$

Design `W1_xor`, `b1_xor`, `W2_xor`, `b2_xor` by hand &mdash; hidden unit 1 computes OR, hidden
unit 2 computes NAND, the output computes AND &mdash; using the soft-AND trick from section 4.
Evaluate with `my_sequential_v` on all four inputs.

*Expected:* predictions `[0, 1, 1, 0]` for inputs `(0,0), (0,1), (1,0), (1,1)`, with every
output below `0.01` or above `0.99`. (Weights of magnitude 20 or so are enough.)

In [ ]:
# Your code here

### Checkpoint 2

The vectorised layer silently depends on `b` having shape `(units,)`. Investigate what happens
otherwise, with `X_demo`, `W1` and `b1` from above.

1. Compute `X_demo @ W1 + b1.reshape(-1, 1)`. What shape is the result, or what error do you
   get? Explain using the broadcasting rules.
2. Now try three examples: `X3 = np.array([[1., 2.], [-1., -1.], [0., 0.]])`. What happens
   with the same column-shaped bias, and why is this case worse?
3. Write `my_dense_safe(A_in, W, b)` that `assert`s the shapes are consistent, with a helpful
   message, before computing.

*Expected:* with two examples, a `ValueError` (shapes `(2,3)` and `(3,1)` do not broadcast);
with three, no error at all &mdash; a `(3, 3)` result in which entry $(i, j)$ received bias
$b_i$ instead of $b_j$. Its first row is `[0.982, 0.982, 0.0003]`
instead of the correct `[0.982, 0.9975, 0.0067]`.

In [ ]:
# Your code here

---

## 8. Recap

- A dense layer is one loop: for each unit, a dot product, a bias, a sigmoid. Store unit $j$'s
  weights as column `W[:, j]`, exactly as TensorFlow does.
- `my_sequential` applies layers in turn. It computes the same function as a TensorFlow model
  loaded with the same weights, to float32 precision.
- Weights can be designed. Each hidden unit of the hand-built roaster is one edge of the
  triangle; the output unit is a soft AND. It classifies `0.995` of the roasts without training.
- $(m \times n)(n \times p) \to (m \times p)$; $(\mathbf{AB})_{ij}$ is row $i$ dotted with column
  $j$; `@` is matrix multiplication and `*` is not.
- Vectorised layer: $\mathbf{A}_{\text{out}} = g(\mathbf{A}_{\text{in}}\mathbf{W} + \mathbf{b})$
  with examples in rows. It agrees with the loop and, even on a laptop CPU, runs dozens of
  times faster.

### Exercises to hand in

1. Sharpen each hidden unit of the hand-designed roaster in turn, multiplying its weights and
   bias by 10. Report the accuracy for each, and explain &mdash; using the hidden activations of
   the misclassified roast &mdash; why sharpening one unit helps and another does not.
2. Change the good-roast region to a **square**: $190 < t < 250$ and $12.5 < d < 14.5$. Design a
   $2 \to 4 \to 1$ network for it by hand, generate labels for `X_cof` from the new rule, and
   report your network's accuracy.
3. Extend `my_dense` and `my_dense_v` to take an `activation` argument accepting `"sigmoid"` and
   `"linear"`. Verify against TensorFlow for both.
4. Time `my_dense_v` for $m$ = 1,000, 10,000, 100,000 and 1,000,000 examples on a
   $2 \to 64$ layer. Plot time against $m$ on log&ndash;log axes and state what the slope tells
   you.
5. Prove that `my_sequential_v` computes, row by row, the same thing as `my_sequential`. One
   paragraph, using the definition of the matrix product.

### Next

**Part 4 &mdash; Practice lab:** a $64 \to 25 \to 15 \to 1$ network that reads handwritten
digits, trained in TensorFlow and then checked, layer by layer, against your own NumPy forward
propagation.